# Análisis de Sensibilidad: KDE vs gridsize — Multi-Kernel

Objetivo: para cada uno de los 6 kernels disponibles, encontrar el primer `gridsize`
(entre 10,000 y 100,000) donde el delta máximo respecto a la referencia sea ≤ 0.001.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
from sklearn.neighbors import KernelDensity
from scipy.stats import gaussian_kde  # solo para referencia Scott/Silverman

In [2]:
USE_GPU = True

import os
from pathlib import Path


def _prepare_cuda_windows():
    """Normaliza CUDA_PATH y registra DLLs para CuPy en Windows."""
    cuda_root = os.environ.get("CUDA_PATH") or os.environ.get("CUDA_PATH_V13_1")
    if not cuda_root:
        return None

    cuda_root = Path(cuda_root)
    if cuda_root.name.lower() == "bin":
        cuda_root = cuda_root.parent

    bin_dir = cuda_root / "bin"
    x64_dir = bin_dir / "x64"
    if not bin_dir.exists():
        raise FileNotFoundError(f"No existe la carpeta CUDA bin: {bin_dir}")

    os.environ["CUDA_PATH"] = str(cuda_root)
    os.add_dll_directory(str(bin_dir))
    if x64_dir.exists():
        os.add_dll_directory(str(x64_dir))
    return cuda_root


_gpu_ok = False
if USE_GPU:
    try:
        _cuda_root = _prepare_cuda_windows()
        if _cuda_root is not None:
            print(f"CUDA_PATH normalizado: {_cuda_root}")

        import cupy as cp
        n_dev = cp.cuda.runtime.getDeviceCount()
        if n_dev == 0:
            raise RuntimeError("No GPUs detected.")
        free_mem, total_mem = cp.cuda.Device(0).mem_info
        print(f"GPU disponible: {n_dev} dispositivo(s)")
        print(f"  Memoria: {free_mem/1e9:.1f} GB libres / {total_mem/1e9:.1f} GB totales")

        # Force a tiny kernel compilation so missing NVRTC/CUDA DLLs fail here.
        _probe = cp.arange(1, dtype=cp.float64)
        _ = cp.asnumpy(_probe + 1.0)
        _gpu_ok = True
    except ImportError:
        print("CuPy no encontrado. Instalalo con: pip install cupy-cuda13x")
        print("Usando CPU.")
    except Exception as e:
        print(f"GPU no disponible ({e}). Usando CPU.")
else:
    print("Modo CPU (USE_GPU=False).")
print(f"
Backend activo: {'GPU (CuPy)' if _gpu_ok else 'CPU (NumPy/sklearn)'}")


CuPy no encontrado. Usando CPU.

Backend activo: CPU (NumPy/sklearn)


In [4]:
FILE_PATH = r"D:\Proyectos\Proyectos Python\Clustering_Microbiota\Datos/otu_data_converted.csv"
df = pd.read_csv(FILE_PATH)
print(f"Shape: {df.shape}")
df.head()

Shape: (441, 4739)


,ID,Otu00001,Otu00002,Otu00003,Otu00004,Otu00005,Otu00006,Otu00007,Otu00008,Otu00009,...,Otu04757,Otu04758,Otu04759,Otu04760,Otu04761,Otu04762,Otu04763,Otu04764,Otu04765,Otu04766
0,MI_001_H,354,817,50,31,448,547,727,353,2,...,0,0,0,0,0,0,0,0,0,0
1,MI_002_H,168,9,131,3005,68,10187,570,2006,104,...,0,0,0,0,0,0,0,0,0,0
2,MI_003_H,19,174,6211,79,1063,0,6077,1852,1471,...,0,0,0,0,0,0,0,0,0,0
3,MI_004_H,264,1816,159,16,110,4,12,206,0,...,0,0,0,0,0,0,0,0,0,0
4,MI_005_H,0,2,0,12,598,39,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [5]:
BW_METHOD     = "scott"   # "scott" | "silverman" | número absoluto
UMBRAL        = 0.001
GRIDSIZE_REF  = 100_000
GRIDSIZE_MIN  = 10_000
GRIDSIZE_STEP = 1_000

KERNELS = ['gaussian', 'epanechnikov', 'tophat', 'exponential', 'linear', 'cosine']

df_num = df.select_dtypes(include=[np.number])
valores = df_num.to_numpy(dtype=float).ravel()
valores = valores[np.isfinite(valores)]
data_kde = valores[valores > 0]

# Compute bandwidth in absolute units
_kde_ref = gaussian_kde(data_kde, bw_method=BW_METHOD if isinstance(BW_METHOD, str) else "scott")
_std = float(np.std(data_kde, ddof=1))
if isinstance(BW_METHOD, str):
    BW_ABS = float(_kde_ref.factor * _std)
else:
    BW_ABS = float(BW_METHOD)

print(f"N positivos: {len(data_kde):,}")
print(f"bw ({BW_METHOD}) = {BW_ABS:.4g}")
print(f"Kernels a evaluar: {KERNELS}")

N positivos: 105,420
bw (scott) = 88.57
Kernels a evaluar: ['gaussian', 'epanechnikov', 'tophat', 'exponential', 'linear', 'cosine']


In [6]:
import time

def make_gpu_eval(kernel_name, data, bw):
    """Returns a GPU-based KDE evaluator for the given kernel."""
    import cupy as cp
    data_gpu = cp.asarray(data, dtype=cp.float64)
    n_data   = len(data_gpu)
    free_mem, _ = cp.cuda.Device(0).mem_info
    chunk = max(100, min(10_000, int(free_mem * 0.20 / (n_data * 8))))

    def kernel_fn(u):
        if kernel_name == "gaussian":
            return cp.exp(-0.5 * u**2) / cp.sqrt(2 * np.pi)
        elif kernel_name == "epanechnikov":
            return cp.where(cp.abs(u) <= 1, 0.75 * (1 - u**2), 0.0)
        elif kernel_name == "tophat":
            return cp.where(cp.abs(u) <= 1, 0.5, 0.0)
        elif kernel_name == "exponential":
            return 0.5 * cp.exp(-cp.abs(u))
        elif kernel_name == "linear":
            return cp.where(cp.abs(u) <= 1, 1.0 - cp.abs(u), 0.0)
        elif kernel_name == "cosine":
            return cp.where(cp.abs(u) <= 1, (np.pi/4) * cp.cos((np.pi/2) * u), 0.0)

    def eval_fn(x_eval):
        x_gpu = cp.asarray(x_eval, dtype=cp.float64)
        y_out = cp.empty(len(x_gpu), dtype=cp.float64)
        for i in range(0, len(x_gpu), chunk):
            xc   = x_gpu[i:i+chunk]
            diff = xc[:, None] - data_gpu[None, :]
            u    = diff / bw
            y_out[i:i+chunk] = cp.sum(kernel_fn(u), axis=1) / (n_data * bw)
        return cp.asnumpy(y_out)
    return eval_fn

def make_cpu_eval(kernel_name, data, bw):
    kde_sk = KernelDensity(kernel=kernel_name, bandwidth=bw)
    kde_sk.fit(data.reshape(-1, 1))
    def eval_fn(x_eval):
        return np.exp(kde_sk.score_samples(x_eval.reshape(-1, 1)))
    return eval_fn

In [ ]:
from tqdm.notebook import tqdm

x_ref = np.logspace(np.log10(data_kde.min()), np.log10(data_kde.max()), GRIDSIZE_REF)
gridsizes = np.arange(GRIDSIZE_MIN, GRIDSIZE_REF, GRIDSIZE_STEP)

results = {}  # kernel -> {deltas, gridsize_opt, delta_opt, y_ref, time_s}

for kernel in KERNELS:
    print(f"\n{'='*55}")
    print(f"Kernel: {kernel}")
    
    if _gpu_ok:
        eval_fn = make_gpu_eval(kernel, data_kde, BW_ABS)
    else:
        eval_fn = make_cpu_eval(kernel, data_kde, BW_ABS)
    
    print("  Calculando referencia...")
    t0 = time.perf_counter()
    y_ref_k = eval_fn(x_ref)
    t_ref = time.perf_counter() - t0
    print(f"  Referencia en {t_ref:.2f}s")
    
    deltas_k = []
    t0 = time.perf_counter()
    for n in tqdm(gridsizes, desc=f"  {kernel}", leave=False):
        x_n      = np.logspace(np.log10(data_kde.min()), np.log10(data_kde.max()), n)
        y_n      = eval_fn(x_n)
        y_interp = np.interp(x_ref, x_n, y_n)
        deltas_k.append(float(np.max(np.abs(y_interp - y_ref_k))))
    
    elapsed = time.perf_counter() - t0
    deltas_k = np.array(deltas_k)
    
    mask_ok = deltas_k <= UMBRAL
    if mask_ok.any():
        idx_opt      = int(np.argmax(mask_ok))
        gridsize_opt = int(gridsizes[idx_opt])
        delta_opt    = float(deltas_k[idx_opt])
    else:
        gridsize_opt = None
        delta_opt    = float(deltas_k.min())
    
    results[kernel] = {
        "deltas": deltas_k,
        "gridsize_opt": gridsize_opt,
        "delta_opt": delta_opt,
        "y_ref": y_ref_k,
        "time_s": elapsed
    }
    
    status = f"gridsize_opt={gridsize_opt:,}" if gridsize_opt else "NO alcanzó umbral"
    print(f"  {kernel}: {status}  |  tiempo={elapsed:.1f}s")

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 15))
fig.suptitle(f"Sensibilidad gridsize — Multi-Kernel\n(bw={BW_ABS:.4g}, umbral δ={UMBRAL})", fontsize=14)

colors_map = {
    'gaussian': 'steelblue', 'epanechnikov': 'crimson', 'tophat': 'darkorange',
    'exponential': 'seagreen', 'linear': 'mediumpurple', 'cosine': 'saddlebrown'
}

for ax, kernel in zip(axes.ravel(), KERNELS):
    res = results[kernel]
    color = colors_map[kernel]
    
    ax.plot(gridsizes, res["deltas"], linewidth=1.5, color=color)
    ax.axhline(UMBRAL, color="tomato", linestyle="--", linewidth=1.5, label=f"Umbral δ={UMBRAL}")
    
    if res["gridsize_opt"] is not None:
        ax.axvline(res["gridsize_opt"], color="green", linestyle=":", linewidth=1.5,
                   label=f"Óptimo: {res['gridsize_opt']:,}")
        ax.scatter([res["gridsize_opt"]], [res["delta_opt"]], color="green", zorder=5, s=60)
    
    ax.set_title(f"{kernel}  (t={res['time_s']:.1f}s)")
    ax.set_xlabel("gridsize")
    ax.set_ylabel("Delta máximo")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

plt.tight_layout()
plt.savefig("kde_gridsize_sensitivity_multikernel.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for kernel, color in colors_map.items():
    ax.plot(x_ref, results[kernel]["y_ref"], color=color, linewidth=1.8, label=kernel)
ax.set_xscale("log")
ax.set_title(f"Comparación de KDE por tipo de kernel  (bw={BW_ABS:.4g})")
ax.set_xlabel("Valor (escala log)")
ax.set_ylabel("Densidad")
ax.legend()
ax.grid(True, alpha=0.3, which="both")
plt.tight_layout()
plt.show()

In [ ]:
print("="*65)
print("RESUMEN — SENSIBILIDAD GRIDSIZE POR KERNEL")
print("="*65)
print(f"{'Kernel':<14} {'gridsize_opt':>14} {'delta_opt':>12} {'tiempo(s)':>12}")
print("-"*65)
for kernel in KERNELS:
    res = results[kernel]
    gs  = f"{res['gridsize_opt']:,}" if res['gridsize_opt'] else "—"
    print(f"  {kernel:<12} {gs:>14} {res['delta_opt']:>12.6f} {res['time_s']:>12.1f}")
print("="*65)

## Sanity check: reproducir exactamente el notebook original para Gaussian

Si el kernel Gaussian del multikernel reproduce el resultado del notebook single-kernel original (`KDE_Gridsize_Sensitivity.ipynb`), confirmamos que toda la maquinaria multikernel es matem?ticamente consistente. Esperamos que `max|y_original ? y_multi(gaussian)|` sea num?ricamente despreciable (~1e-10 en CPU, puede ser algo mayor en GPU por precisi?n float).


In [ ]:
# Sanity check: multikernel(gaussian) vs scipy.gaussian_kde (el notebook original)
from scipy.stats import gaussian_kde as _gk
_kde_orig = _gk(data_kde, bw_method=BW_METHOD if isinstance(BW_METHOD, str) else "scott")
_y_orig   = _kde_orig(x_ref)
_y_multi  = results['gaussian']['y_ref']

_max_diff = float(np.max(np.abs(_y_orig - _y_multi)))
_rel_diff = _max_diff / float(np.max(_y_orig))

print("="*60)
print("SANITY CHECK ? Gaussian multikernel vs scipy.gaussian_kde")
print("="*60)
print(f"  max|orig - multi|          = {_max_diff:.2e}")
print(f"  diferencia relativa (%)    = {100*_rel_diff:.2e}")
print(f"  y_orig  max = {_y_orig.max():.6g}")
print(f"  y_multi max = {_y_multi.max():.6g}")
print("="*60)
assert _rel_diff < 1e-4, (
    f"Discrepancia inesperada entre original y multikernel para Gaussian: "
    f"rel_diff={_rel_diff:.2e}"
)
print("OK: el kernel gaussian del multikernel reproduce el resultado del notebook original.")
